<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/08-visualization/02_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 8: Exercises

Twelve charts to build: four Easy, five Medium and three Hard. Each is a function that draws a chart and returns the figure, which is how plotting code is usually organised in real projects.

How do you test a chart automatically? The checks open up the figure your function returns and inspect it: how many bars there are and in what order, which colour each group got, where the axis starts, whether there's a second y-axis, what text is in each cell. When a check passes, it draws one example with your function so you can see the result.

**Run the setup cell first.** It applies the course plotting style from the lesson, so `PALETTE`, `SURFACE`, `INK`, `INK_2`, `MUTED`, `AXIS`, `SEQUENTIAL` and `DIVERGING` are all available.

Rules:

- Use the object-oriented style (`fig, ax = plt.subplots()`), and return the figure as described. Don't call `plt.show()` inside your functions.
- Matplotlib, NumPy and pandas only.
- Solutions are in `03_solutions.ipynb`.

In [ ]:
# Setup: run this cell once before you start.
# It sets up the course plotting style and defines the check functions.
# Checks close the figures they inspect; after a PASS they draw one example
# with your function so you can see it.

import copy
import math
import inspect
import tempfile
import time
from pathlib import Path

import numpy as np
import pandas as pd
from io import StringIO


import matplotlib.colors as mcolors

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# Categorical colours, always used in this order. The first three are checked to stay
# distinguishable for the common forms of colour blindness, even in scatter plots.
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]

SURFACE = "#fcfcfb"     # chart background
INK = "#0b0b0b"         # titles and important text
INK_2 = "#52514e"       # labels and tick text
MUTED = "#898781"       # de-emphasised marks
GRID = "#e1e0d9"        # gridlines
AXIS = "#c3c2b7"        # axis lines

# Sequential: one hue, light to dark, for magnitude (heatmaps, counts)
SEQUENTIAL = LinearSegmentedColormap.from_list(
    "course_blues", ["#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])
# Diverging: two opposite hues with a neutral grey in the middle, for values around zero
DIVERGING = LinearSegmentedColormap.from_list(
    "course_diverging", ["#184f95", "#86b6ef", "#f0efec", "#f0a3a2", "#c23a3a"])


def use_course_style():
    for cmap in (SEQUENTIAL, DIVERGING):
        if cmap.name not in mpl.colormaps:
            mpl.colormaps.register(cmap)
    mpl.rcParams.update({
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "savefig.facecolor": SURFACE,
        "figure.dpi": 100,
        "savefig.dpi": 150,
        "figure.figsize": (7, 4),
        "font.size": 10,
        "text.color": INK,
        "axes.titlesize": 12,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        "axes.titlepad": 10,
        "axes.labelcolor": INK_2,
        "axes.edgecolor": AXIS,
        "axes.linewidth": 0.8,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "axes.axisbelow": True,
        "grid.color": GRID,
        "grid.linewidth": 0.8,
        "grid.linestyle": "-",
        "xtick.color": AXIS,
        "ytick.color": AXIS,
        "xtick.labelcolor": INK_2,
        "ytick.labelcolor": INK_2,
        "axes.prop_cycle": mpl.cycler(color=PALETTE),
        "lines.linewidth": 1.6,
        "lines.solid_capstyle": "round",
        "lines.markersize": 6,
        "patch.linewidth": 0,
        "legend.frameon": False,
        "legend.labelcolor": INK_2,
        "image.cmap": "course_blues",
    })


use_course_style()


class Raises:
    """Marks a test case where the code is expected to raise an error."""
    def __init__(self, exc):
        self.exc = exc

    def __repr__(self):
        return f"{self.exc.__name__} to be raised"


def _same(a, b, tol=1e-6):
    if isinstance(a, (pd.DataFrame, pd.Series)) or isinstance(b, (pd.DataFrame, pd.Series)):
        if type(a) is not type(b):
            return False
        options = dict(check_dtype=False, check_exact=False, rtol=tol, atol=tol, check_index_type=False)
        try:
            if isinstance(a, pd.DataFrame):
                pd.testing.assert_frame_equal(a, b, check_column_type=False, **options)
            else:
                pd.testing.assert_series_equal(a, b, **options)
            return True
        except AssertionError:
            return False
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        try:
            a_arr, b_arr = np.asarray(a), np.asarray(b)
        except Exception:
            return False
        if a_arr.shape != b_arr.shape:
            return False
        if a_arr.dtype.kind in "fc" or b_arr.dtype.kind in "fc":
            return bool(np.allclose(a_arr, b_arr, rtol=tol, atol=tol))
        return bool(np.array_equal(a_arr, b_arr))
    if isinstance(a, (float, np.floating)) or isinstance(b, (float, np.floating)):
        if not isinstance(a, (int, float, np.number)) or not isinstance(b, (int, float, np.number)):
            return False
        if math.isnan(a) or math.isnan(b):
            return False
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b))
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k]) for k in a)
    return a == b


def _short(x, limit=90):
    if isinstance(x, (pd.DataFrame, pd.Series)):
        lines = (x.to_string(max_rows=6, max_cols=6) if isinstance(x, pd.DataFrame) else x.to_string(max_rows=6)).splitlines()
        return f"{type(x).__name__} shape={x.shape}\n          " + "\n          ".join(lines[:8])
    if isinstance(x, np.ndarray):
        body = np.array2string(x, precision=4, threshold=12, edgeitems=3, separator=", ")
        text = f"array(shape={x.shape}, dtype={x.dtype}, {' '.join(body.split())})"
    elif isinstance(x, tuple) and any(isinstance(i, np.ndarray) for i in x):
        text = "(" + ", ".join(_short(i, limit) for i in x) + ")"
        return text
    else:
        text = repr(x)
    if isinstance(x, (pd.DataFrame, pd.Series)):
        return text
    return text if len(text) <= limit else text[:limit - 3] + "..."


def _scenarios(name, cases):
    for description, scenario, expected, hint in cases:
        try:
            result = scenario()
            plt.close("all")
        except Exception as e:
            plt.close("all")
            if isinstance(expected, Raises) and isinstance(e, expected.exc):
                continue
            print(f"FAIL  {name}: {description}")
            print(f"      raised {type(e).__name__}: {e}")
            if isinstance(expected, Raises):
                print(f"      expected {expected!r}")
            else:
                print(f"      expected: {_short(expected)}")
            if hint:
                print(f"      hint: {hint}")
            return False
        if isinstance(expected, Raises):
            print(f"FAIL  {name}: {description}")
            print(f"      expected {expected!r}, but nothing was raised")
            if hint:
                print(f"      hint: {hint}")
            return False
        if not _same(result, expected):
            print(f"FAIL  {name}: {description}")
            print(f"      expected: {_short(expected)}")
            print(f"      got:      {_short(result)}")
            if hint:
                print(f"      hint: {hint}")
            return False
    print(f"PASS  all {len(cases)} tests passed for {name}")
    return True


def _hex(colour):
    return mcolors.to_hex(colour).lower()


def _title(ax):
    """The axes title, wherever it was placed (the house style puts titles on the left)."""
    return next((ax.get_title(loc) for loc in ("left", "center", "right") if ax.get_title(loc)), "")


def _data_lines(ax):
    """Lines that carry data (skips axhline/axvline reference lines)."""
    return [l for l in ax.get_lines() if len(np.atleast_1d(l.get_xdata())) > 2]


def _sample_penguins():
    rng = np.random.default_rng(3)
    rows = []
    for sp, (bl, mass) in {"Adelie": (39, 3700), "Chinstrap": (49, 3730), "Gentoo": (47, 5080)}.items():
        for _ in range(30):
            rows.append({"species": sp, "bill_length_mm": rng.normal(bl, 2.5),
                         "body_mass_g": rng.normal(mass, 350), "flipper_length_mm": rng.normal(200, 8)})
    return pd.DataFrame(rows)


# ---- checks ----

def check_line_chart(f):
    x = np.arange(10)
    y = x ** 2

    def info():
        fig, ax = f(x, y, "Squares grow fast", "n", "n squared")
        lines = _data_lines(ax)
        return (len(lines), _title(ax), ax.get_xlabel(), ax.get_ylabel())

    def data():
        fig, ax = f(x, y, "t", "x", "y")
        line = _data_lines(ax)[0]
        return (np.asarray(line.get_xdata()), np.asarray(line.get_ydata()))

    def style():
        fig, ax = f(x, y, "t", "x", "y")
        return (_hex(_data_lines(ax)[0].get_color()), ax.get_legend() is None)

    passed = _scenarios("line_chart", [
        ("returns (fig, ax) with one line, a title and axis labels", info,
         (1, "Squares grow fast", "n", "n squared"), "fig, ax = plt.subplots(); ax.plot(...); ax.set_title(...); return fig, ax"),
        ("plots the data given", data, (x, y), None),
        ("first palette colour, no legend for a single series", style, (PALETTE[0], True),
         "one series doesn't need a legend: the title says what it is"),
    ])
    if passed:
        f(x, y, "Example: squares grow fast", "n", "n squared")


def check_histogram(f):
    values = np.array([1.0, 2.0, 2.5, 3.0, np.nan, 3.5, 4.0, 8.0, np.nan])

    def bars():
        fig, ax = f(values, 5, "value")
        heights = [p.get_height() for p in ax.patches]
        return (len(ax.patches), sum(heights))

    def labels():
        fig, ax = f(values, 5, "Body mass (g)")
        return (ax.get_xlabel(), ax.get_ylabel())

    passed = _scenarios("histogram", [
        ("missing values are dropped, every other value counted once", bars, (5, 7),
         "np.histogram can't work out a range when there are NaNs: drop them first"),
        ("axis labels", labels, ("Body mass (g)", "Count"), "set the y label to 'Count'"),
    ])
    if passed:
        f(np.random.default_rng(0).normal(4200, 800, 300), 20, "Body mass (g)")


def check_sorted_barh(f):
    labels = ["apples", "bananas", "cherries", "dates"]
    values = [30.0, 10.0, 20.0, 5.0]

    def order():
        fig, ax = f(labels, values, "Mean body mass (g)")
        bars = sorted(ax.patches, key=lambda p: p.get_y(), reverse=True)    # top bar first
        return [round(b.get_width()) for b in bars]

    def tick_labels():
        fig, ax = f(labels, values, "Mean body mass (g)")
        ticks = sorted(zip(ax.get_yticks(), [t.get_text() for t in ax.get_yticklabels()]), reverse=True)
        return [text for _, text in ticks]

    def honest():
        fig, ax = f(labels, values, "Mean body mass (g)")
        colours = {_hex(p.get_facecolor()) for p in ax.patches}
        return (ax.get_xlim()[0], len(colours), len(ax.texts))

    passed = _scenarios("sorted_barh", [
        ("largest bar at the top", order, [30, 20, 10, 5],
         "sort ascending before calling barh: barh draws the first bar at the bottom"),
        ("labels move with their bars", tick_labels, ["apples", "cherries", "bananas", "dates"],
         "reorder the labels with the same positions as the values (np.argsort), don't sort them separately"),
        ("starts at zero, one colour, a value label per bar", honest, (0.0, 1, 4),
         "ax.set_xlim(left=0); one colour for one measure; ax.text(...) at each bar end"),
    ])
    if passed:
        f(["Adelie", "Gentoo", "Chinstrap"], [3701.0, 5076.0, 3733.0], "Mean body mass (g)")


def check_save_figure(f):
    def run(formats):
        fig, ax = plt.subplots()
        ax.plot([1, 2, 3])
        folder = Path(tempfile.mkdtemp()) / "reports" / "figs"
        paths = f(fig, folder / "chart", formats)
        return fig, [Path(p) for p in paths]

    def png():
        fig, paths = run(("png",))
        return (len(paths), paths[0].suffix, paths[0].read_bytes()[:8] == b"\x89PNG\r\n\x1a\n")

    def closed():
        fig, paths = run(("png",))
        return plt.fignum_exists(fig.number)

    _scenarios("save_figure", [
        ("creates folders and a real PNG", png, (1, ".png", True), "path.parent.mkdir(parents=True, exist_ok=True)"),
        ("one file per format", lambda: sorted(p.suffix for p in run(("png", "svg"))[1]), [".png", ".svg"],
         "add the suffix with path.with_suffix('.' + fmt)"),
        ("closes the figure afterwards", closed, False, "plt.close(fig)"),
    ])


def check_scatter_by_group(f):
    df = _sample_penguins()
    order = ["Adelie", "Chinstrap", "Gentoo"]

    def colours(data):
        fig, ax = f(data, "bill_length_mm", "body_mass_g", "species", order)
        legend = ax.get_legend()
        names = [t.get_text() for t in legend.get_texts()] if legend else []
        return {name: _hex(c.get_facecolor()[0]) for name, c in zip(names, ax.collections)}

    def markers():
        fig, ax = f(df, "bill_length_mm", "body_mass_g", "species", order)
        return len({tuple(np.round(c.get_paths()[0].vertices.ravel()[:8], 3)) for c in ax.collections})

    def labels():
        fig, ax = f(df, "bill_length_mm", "body_mass_g", "species", order)
        return (ax.get_xlabel(), ax.get_ylabel(), len(ax.collections))

    passed = _scenarios("scatter_by_group", [
        ("one scatter per group, colours in order, with a legend", lambda: colours(df),
         {"Adelie": PALETTE[0], "Chinstrap": PALETTE[1], "Gentoo": PALETTE[2]},
         "loop over `order`, use PALETTE[i] for order[i], and pass label=group_name"),
        ("colour follows the group, not its position", lambda: colours(df[df["species"] != "Adelie"]),
         {"Chinstrap": PALETTE[1], "Gentoo": PALETTE[2]},
         "pick each group's colour from its position in `order`, and skip groups with no rows"),
        ("each group has its own marker shape", markers, 3, "colour shouldn't be the only way to tell groups apart"),
        ("axis labels and three groups", labels, ("bill_length_mm", "body_mass_g", 3), None),
    ])
    if passed:
        f(df, "bill_length_mm", "body_mass_g", "species", order)


def check_small_multiples(f):
    df = _sample_penguins()
    order = ["Adelie", "Chinstrap", "Gentoo"]

    def layout():
        fig = f(df, "body_mass_g", "species", order)
        return len(fig.axes)

    def shared():
        fig = f(df, "body_mass_g", "species", order)
        a, b = fig.axes[0], fig.axes[-1]
        return (a.get_xlim() == b.get_xlim(), a.get_ylim() == b.get_ylim())

    def same_bins():
        fig = f(df, "body_mass_g", "species", order)
        edges = [tuple(np.round(sorted({p.get_x() for p in ax.patches}), 6)) for ax in fig.axes]
        return len(set(edges)) == 1

    def colours():
        fig = f(df, "body_mass_g", "species", order)
        return [_hex(ax.patches[0].get_facecolor()) for ax in fig.axes]

    passed = _scenarios("small_multiples", [
        ("one panel per group", layout, 3, None),
        ("panels share both axes", shared, (True, True), "plt.subplots(n, 1, sharex=True, sharey=True)"),
        ("every panel uses the same bin edges", same_bins, True, "compute bins once with np.linspace over the whole column"),
        ("each panel in its group's colour", colours, PALETTE[:3], None),
    ])
    if passed:
        f(df, "body_mass_g", "species", order)


def check_learning_curves(f):
    epochs = np.arange(1, 21)
    train = 1.0 / epochs
    val = 0.6 / np.sqrt(epochs) + 0.004 * (epochs - 8) ** 2 / 10
    history = pd.DataFrame({"epoch": epochs, "train_loss": train, "val_loss": val})
    best_epoch = int(history.loc[history["val_loss"].idxmin(), "epoch"])

    def lines():
        fig, ax = f(history)
        named = {l.get_label(): l for l in _data_lines(ax)}
        return sorted(named)

    def best_marker():
        fig, ax = f(history)
        vertical = [l for l in ax.get_lines() if len(set(np.atleast_1d(l.get_xdata()))) == 1]
        return float(np.atleast_1d(vertical[0].get_xdata())[0]) if vertical else None

    def labels():
        fig, ax = f(history)
        return (ax.get_xlabel(), ax.get_ylabel(), ax.get_legend() is not None)

    passed = _scenarios("learning_curves", [
        ("a train line and a validation line", lines, ["train", "validation"], "label='train' and label='validation'"),
        ("a vertical line at the best validation epoch", best_marker, float(best_epoch),
         "ax.axvline(epoch_with_lowest_val_loss)"),
        ("labels and a legend", labels, ("Epoch", "Loss", True), None),
    ])
    if passed:
        f(history)


def check_confusion_heatmap(f):
    cm = np.array([[38, 0, 0], [2, 15, 0], [0, 0, 31]])
    labels = ["Adelie", "Chinstrap", "Gentoo"]

    def image():
        fig, ax = f(cm, labels)
        img = ax.get_images()[0]
        return (np.asarray(img.get_array()), img.get_cmap().name)

    def numbers():
        fig, ax = f(cm, labels)
        return sorted(int(t.get_text()) for t in ax.texts)

    def text_colours():
        fig, ax = f(cm, labels)
        by_pos = {(round(t.get_position()[1]), round(t.get_position()[0])): _hex(t.get_color()) for t in ax.texts}
        return (by_pos[(0, 0)], by_pos[(1, 0)])

    def ticks():
        fig, ax = f(cm, labels)
        return ([t.get_text() for t in ax.get_xticklabels()], [t.get_text() for t in ax.get_yticklabels()],
                ax.get_xlabel(), ax.get_ylabel())

    passed = _scenarios("confusion_heatmap", [
        ("the matrix as an image, with the sequential colour scale", image, (cm, "course_blues"),
         "ax.imshow(cm, cmap=SEQUENTIAL): counts are a magnitude"),
        ("every count written in its cell", numbers, sorted(cm.ravel().tolist()), None),
        ("white text on dark cells, ink on light ones", text_colours, ("#ffffff", INK),
         "use white when the count is above half the maximum, INK otherwise"),
        ("class names on both axes", ticks, (labels, labels, "Predicted", "True"), None),
    ])
    if passed:
        f(cm, labels)


def check_indexed_chart(f):
    idx = pd.date_range("2025-01-01", periods=6, freq="MS")
    df = pd.DataFrame({"visits": [1000, 1100, 1050, 1200, 1300, 1400], "revenue": [20.0, 25, 24, 30, 36, 44]}, index=idx)

    def one_axis():
        fig, ax = f(df)
        return len(fig.axes)

    def starts():
        fig, ax = f(df)
        return sorted((l.get_label(), round(float(np.asarray(l.get_ydata())[0]), 6), round(float(np.asarray(l.get_ydata())[-1]), 6))
                      for l in _data_lines(ax))

    def reference():
        fig, ax = f(df)
        flat = [l for l in ax.get_lines() if len(set(np.round(np.atleast_1d(l.get_ydata()).astype(float), 9))) == 1]
        return (bool(flat) and float(np.atleast_1d(flat[0].get_ydata())[0]) == 100.0, "100" in ax.get_ylabel())

    passed = _scenarios("indexed_chart", [
        ("one axes only: no second y-axis", one_axis, 1, "don't use ax.twinx(): index both series instead"),
        ("each series starts at 100", starts, [("revenue", 100.0, 220.0), ("visits", 100.0, 140.0)],
         "divide each column by its first value and multiply by 100"),
        ("a reference line at 100, and the y label says so", reference, (True, True),
         "ax.axhline(100, ...) and a y label like 'Index (first month = 100)'"),
    ])
    if passed:
        f(df)


def check_correlation_heatmap(f):
    rng = np.random.default_rng(4)
    a = rng.normal(size=200)
    df = pd.DataFrame({"a": a, "b": 0.8 * a + rng.normal(0, 0.6, 200), "c": rng.normal(size=200), "d": -a + rng.normal(0, 0.3, 200)})

    def image():
        fig, ax = f(df)
        img = ax.get_images()[0]
        return (np.asarray(img.get_array()), img.get_cmap().name, img.get_clim())

    def labels():
        fig, ax = f(df)
        return ([t.get_text() for t in ax.get_xticklabels()], [t.get_text() for t in ax.get_yticklabels()], len(ax.texts))

    passed = _scenarios("correlation_heatmap", [
        ("diverging colours, fixed from -1 to 1", image, (df.corr().to_numpy(), "course_diverging", (-1.0, 1.0)),
         "ax.imshow(corr, cmap=DIVERGING, vmin=-1, vmax=1): without vmin/vmax a weak correlation can look strong"),
        ("column names on both axes, a number in every cell", labels, (list(df.columns), list(df.columns), 16), None),
    ])
    if passed:
        f(df)


def check_decision_regions(f):
    rng = np.random.default_rng(5)
    X = np.vstack([rng.normal([0, 0], 0.7, size=(40, 2)), rng.normal([3, 3], 0.7, size=(40, 2))])
    y = np.repeat([0, 1], 40)
    calls = []

    def predict(points):
        calls.append(np.asarray(points).shape)
        return (np.asarray(points).sum(axis=1) > 3).astype(int)

    def vectorized():
        calls.clear()
        f(predict, X, y, ["low", "high"], resolution=50)
        return calls

    def covers():
        fig, ax = f(predict, X, y, ["low", "high"], resolution=50)
        x0, x1 = ax.get_xlim()
        y0, y1 = ax.get_ylim()
        return (x0 < X[:, 0].min(), x1 > X[:, 0].max(), y0 < X[:, 1].min(), y1 > X[:, 1].max())

    def layers():
        fig, ax = f(predict, X, y, ["low", "high"], resolution=50)
        legend = ax.get_legend()
        return (len(ax.collections) >= 3, sorted(t.get_text() for t in legend.get_texts()) if legend else None)

    passed = _scenarios("decision_regions", [
        ("predict is called once, on the whole grid", vectorized, [(2500, 2)],
         "build the grid with np.meshgrid, stack it into an (n, 2) array, and predict everything in one call"),
        ("the grid extends past the data", covers, (True, True, True, True), "pad the range on every side"),
        ("filled regions plus one scatter per class, with a legend", layers, (True, ["high", "low"]),
         "ax.contourf for the regions, then ax.scatter for each class with label=labels[k]"),
    ])
    if passed:
        f(predict, X, y, ["low", "high"])


def check_rolling_band(f):
    idx = pd.date_range("2025-01-01", periods=120, freq="D")
    series = pd.Series(np.sin(np.arange(120) / 9) * 10 + np.random.default_rng(6).normal(0, 2, 120) + 50, index=idx)
    expected_mean = series.rolling(14, center=True).mean().to_numpy()

    def lines():
        fig, ax = f(series, 14)
        data = _data_lines(ax)
        alphas = sorted((l.get_alpha() if l.get_alpha() is not None else 1.0) for l in data)
        return (len(data), alphas[0] < 1.0)

    def mean_line():
        fig, ax = f(series, 14)
        for l in _data_lines(ax):
            yd = np.asarray(l.get_ydata(), dtype=float)
            if np.isnan(yd).any():
                return bool(np.allclose(yd, expected_mean, equal_nan=True))
        return "no line with the rolling mean (it should start and end with NaN)"

    def band():
        fig, ax = f(series, 14)
        return any(type(c).__name__ in ("PolyCollection", "FillBetweenPolyCollection") for c in ax.collections)

    passed = _scenarios("rolling_band", [
        ("a faint raw line and a solid average line", lines, (2, True), "draw the raw data with alpha around 0.3"),
        ("the average is a centred rolling mean", mean_line, True, "series.rolling(window, center=True).mean()"),
        ("a shaded band for mean ± one standard deviation", band, True, "ax.fill_between(index, mean - std, mean + std, alpha=0.15)"),
    ])
    if passed:
        f(series, 14)


print("Setup done. Checks are ready.")

## Easy

### Exercise 1: `line_chart`

Write `line_chart(x, y, title, xlabel, ylabel)` that draws a single line and returns `(fig, ax)`.

- Use the object-oriented style: `fig, ax = plt.subplots()`.
- Set the title and both axis labels.
- Use the first palette colour. A single series doesn't need a legend.

Don't call `plt.show()` inside the function. Return the figure so the caller decides what to do with it. That rule applies to every exercise in this notebook.

In [ ]:
def line_chart(x, y, title: str, xlabel: str, ylabel: str):
    # your code here
    pass

In [ ]:
check_line_chart(line_chart)

### Exercise 2: `histogram`

Write `histogram(values, bins, xlabel)` that draws a histogram of a NumPy array and returns `(fig, ax)`.

- The array may contain `NaN`. Leave those values out, and count every other value exactly once.
- Use `bins` bins, the first palette colour, and a thin background-coloured edge so adjacent bars stay visually separate.
- Label the x axis with `xlabel` and the y axis with `"Count"`.

In [ ]:
def histogram(values, bins: int, xlabel: str):
    # your code here
    pass

In [ ]:
check_histogram(histogram)

### Exercise 3: `sorted_barh`

Write `sorted_barh(labels, values, xlabel)` that draws a horizontal bar chart and returns `(fig, ax)`:

- The largest value at the **top**, and each label staying with its own bar.
- The x axis starting at **zero**.
- One colour for all bars (they show a single measure).
- Each bar's value written just past its end.

Careful: `barh` draws its first bar at the bottom.

In [ ]:
def sorted_barh(labels, values, xlabel: str):
    # your code here
    pass

In [ ]:
check_sorted_barh(sorted_barh)

### Exercise 4: `save_figure`

Write `save_figure(fig, path, formats=("png",))` that saves a figure for a report:

- `path` has no extension, like `"reports/figures/accuracy"`. Save one file per format, adding the right extension (`accuracy.png`, `accuracy.svg`).
- Create any missing folders.
- Trim empty space around the figure.
- Close the figure afterwards, to free its memory.
- Return the list of saved paths.

In [ ]:
from pathlib import Path


def save_figure(fig, path, formats=("png",)) -> list:
    # your code here
    pass

In [ ]:
check_save_figure(save_figure)

## Medium

### Exercise 5: `scatter_by_group`

Write `scatter_by_group(df, x, y, group, order)` that draws a scatter plot of two columns with one colour **and** one marker shape per group, and returns `(fig, ax)`.

- `order` lists every possible group, like `["Adelie", "Chinstrap", "Gentoo"]`. Group `order[i]` always gets `PALETTE[i]`, **even if some groups are missing from `df`**. That way Chinstrap is orange in every chart, filtered or not.
- Skip groups that have no rows.
- Add a legend with the group names, and label the axes with the column names.

In [ ]:
MARKER_SHAPES = ["o", "s", "^", "D", "v", "P", "X", "*"]


def scatter_by_group(df, x: str, y: str, group: str, order: list):
    # your code here
    pass

In [ ]:
check_scatter_by_group(scatter_by_group)

### Exercise 6: `small_multiples`

Write `small_multiples(df, column, group, order, bins=20)` that draws one histogram of `column` per group, stacked vertically, and returns the **figure**.

- One panel per group, in `order`.
- All panels share **both** axes, so heights and positions can be compared directly.
- All panels use **the same bin edges**, computed once from the whole column.
- Each panel uses its group's palette colour and shows the group's name.

In [ ]:
def small_multiples(df, column: str, group: str, order: list, bins: int = 20):
    # your code here
    pass

In [ ]:
check_small_multiples(small_multiples)

### Exercise 7: `learning_curves`

Write `learning_curves(history)` for a DataFrame with columns `epoch`, `train_loss` and `val_loss`. Return `(fig, ax)` with:

- a line labelled `"train"` and a line labelled `"validation"`
- a vertical line at the epoch with the **lowest validation loss**, the epoch early stopping would keep
- `"Epoch"` and `"Loss"` as axis labels, and a legend

This is the chart you'd look at after every training run in Modules 6 and beyond.

In [ ]:
def learning_curves(history):
    # your code here
    pass

In [ ]:
check_learning_curves(learning_curves)

### Exercise 8: `confusion_heatmap`

Write `confusion_heatmap(cm, labels)` that draws a confusion matrix (rows are true classes, columns are predictions) and returns `(fig, ax)`:

- Draw the matrix with `imshow`, using the **sequential** colour scale (`SEQUENTIAL`), since counts are a magnitude.
- Write the count in every cell: **white** on dark cells (count above half the maximum), `INK` otherwise.
- Class names on both axes, with `"Predicted"` as the x label and `"True"` as the y label.

In [ ]:
def confusion_heatmap(cm, labels: list):
    # your code here
    pass

In [ ]:
check_confusion_heatmap(confusion_heatmap)

### Exercise 9: `indexed_chart`

Two measures with different units, like visits and revenue, are often shown with two y-axes. That's misleading, because the chart maker decides how the two scales line up. Write `indexed_chart(df)` that does the honest version instead, and returns `(fig, ax)`:

- `df` has a date index and one numeric column per series.
- Divide each column by its **first** value and multiply by 100, so every series starts at 100.
- Plot every series on **one** axes, labelled with its column name, with a legend.
- Add a horizontal reference line at 100, and a y label that mentions 100, such as `"Index (first month = 100)"`.

In [ ]:
def indexed_chart(df):
    # your code here
    pass

In [ ]:
check_indexed_chart(indexed_chart)

## Hard

### Exercise 10: `correlation_heatmap`

Write `correlation_heatmap(df)` that draws the correlation matrix of a DataFrame's columns and returns `(fig, ax)`:

- Use the **diverging** colour scale (`DIVERGING`), fixed to run from **-1 to 1**, whatever values happen to be present.
- Write the correlation, to two decimals, in every cell, choosing white or `INK` text for readability.
- Column names on both axes.
- Add a colour bar.

In [ ]:
def correlation_heatmap(df):
    # your code here
    pass

In [ ]:
check_correlation_heatmap(correlation_heatmap)

### Exercise 11: `decision_regions`

Write `decision_regions(predict, X, y, labels, resolution=200)` that shows how a classifier divides up a 2D feature space, and returns `(fig, ax)`. In earlier modules we drew this with `#` and `.` characters.

- `predict` takes an array of shape `(n, 2)` and returns `n` integer class labels.
- Build a `resolution × resolution` grid covering the data, **padded by 10% of the range on every side**, and call `predict` **once** on the whole grid.
- Draw the predicted regions with `ax.contourf`, faintly, using the palette colours.
- Draw the training points on top, one scatter per class, coloured by class, labelled `labels[k]` for class `k`, with a legend.

In [ ]:
def decision_regions(predict, X, y, labels: list, resolution: int = 200):
    # your code here
    pass

In [ ]:
check_decision_regions(decision_regions)

### Exercise 12: `rolling_band`

Write `rolling_band(series, window)` for a pandas Series with a date index. Return `(fig, ax)` showing:

- the raw values as a faint line (`alpha` below 1)
- the **centred** rolling mean over `window` points as a solid line
- a shaded band from (rolling mean − rolling standard deviation) to (rolling mean + rolling standard deviation)

A band like this shows both the trend and how much the values typically vary around it.

In [ ]:
def rolling_band(series, window: int):
    # your code here
    pass

In [ ]:
check_rolling_band(rolling_band)

## Done?

Once everything passes, compare with `03_solutions.ipynb`. Then head to the mini project in `04_mini_project.ipynb`, where several of these charts come together in a visual report on a penguin classifier.